<a href="https://colab.research.google.com/github/Zforeror/Tareas-Matem-ticas-de-aprendizaje-de-m-quina/blob/main/Support_Vector_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SUPPORT VECTOR MACHINE MEDIANTE OPTIMIZACIÓN CONVEXA
# Código completo para Google Colab
# ============================================================

# Instalar las librerías necesarias
!pip install -q cvxpy scikit-learn matplotlib

# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import make_blobs
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ============================================================
# 2. CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================

SEMILLA = 42
C = 1.0

np.random.seed(SEMILLA)


# ============================================================
# 3. GENERACIÓN DE LOS DATOS
# ============================================================

# Se generan dos grupos de datos en dos dimensiones.
X, y_original = make_blobs(
    n_samples=200,
    centers=[(-2, -2), (2, 2)],
    cluster_std=1.5,
    random_state=SEMILLA
)

# La SVM utiliza etiquetas -1 y +1.
y = np.where(y_original == 0, -1, 1)

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)
print("Etiquetas utilizadas:", np.unique(y))


# ============================================================
# 4. DIVISIÓN ENTRE ENTRENAMIENTO Y PRUEBA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEMILLA,
    stratify=y
)

print("\nDatos de entrenamiento:", len(X_train))
print("Datos de prueba:", len(X_test))


# ============================================================
# 5. NORMALIZACIÓN DE LOS DATOS
# ============================================================

# La normalización ayuda a que ambas características tengan
# escalas comparables.
escalador = StandardScaler()

X_train_scaled = escalador.fit_transform(X_train)
X_test_scaled = escalador.transform(X_test)
X_scaled = escalador.transform(X)


# ============================================================
# 6. FORMULACIÓN DEL PROBLEMA DE OPTIMIZACIÓN CONVEXA
# ============================================================

n_muestras, n_caracteristicas = X_train_scaled.shape

# Variables del problema:
# w  = vector de pesos
# b  = sesgo
# xi = variables de holgura

w = cp.Variable(n_caracteristicas)
b = cp.Variable()
xi = cp.Variable(n_muestras, nonneg=True)

# Función objetivo:
#
# minimizar  1/2 ||w||² + C sum(xi)
#
# El primer término busca un margen grande.
# El segundo término penaliza los errores de clasificación.

funcion_objetivo = cp.Minimize(
    0.5 * cp.sum_squares(w) + C * cp.sum(xi)
)

# Restricciones:
#
# y_i (w^T x_i + b) >= 1 - xi_i
# xi_i >= 0
#
# xi >= 0 ya se estableció cuando se creó la variable.

restriccion_margen = cp.multiply(
    y_train,
    X_train_scaled @ w + b
) >= 1 - xi

restricciones = [restriccion_margen]

problema = cp.Problem(
    funcion_objetivo,
    restricciones
)


# ============================================================
# 7. SOLUCIÓN DEL PROBLEMA
# ============================================================

# Se intenta resolver con CLARABEL.
# En caso de error se utiliza SCS.

try:
    problema.solve(solver=cp.CLARABEL)
except Exception:
    problema.solve(solver=cp.SCS)

print("\n" + "=" * 60)
print("RESULTADOS DE LA OPTIMIZACIÓN")
print("=" * 60)

print("Estado del problema:", problema.status)
print("Valor óptimo:", problema.value)
print("Vector de pesos w:", w.value)
print("Sesgo b:", b.value)


# ============================================================
# 8. FUNCIÓN DE PREDICCIÓN
# ============================================================

def predecir(X_datos):
    """
    Clasifica cada punto utilizando el signo de:

                  w^T x + b
    """

    valores = X_datos @ w.value + b.value

    return np.where(valores >= 0, 1, -1)


# Predicciones sobre los conjuntos de entrenamiento y prueba
y_pred_train = predecir(X_train_scaled)
y_pred_test = predecir(X_test_scaled)


# ============================================================
# 9. EVALUACIÓN DEL MODELO
# ============================================================

accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)

print("\n" + "=" * 60)
print("EVALUACIÓN DEL MODELO")
print("=" * 60)

print(f"Exactitud en entrenamiento: {accuracy_train:.2%}")
print(f"Exactitud en prueba:        {accuracy_test:.2%}")

print("\nReporte de clasificación:")
print(
    classification_report(
        y_test,
        y_pred_test,
        target_names=["Clase -1", "Clase +1"],
        digits=4
    )
)


# ============================================================
# 10. CÁLCULO DEL MARGEN
# ============================================================

norma_w = np.linalg.norm(w.value)

if norma_w > 0:
    ancho_margen = 2 / norma_w
else:
    ancho_margen = np.inf

print("Norma de w:", norma_w)
print("Ancho total del margen:", ancho_margen)


# ============================================================
# 11. IDENTIFICACIÓN DE LOS VECTORES DE SOPORTE
# ============================================================

# Los multiplicadores duales asociados a la restricción
# permiten identificar los vectores de soporte.

multiplicadores_duales = restriccion_margen.dual_value

indices_soporte = np.where(
    multiplicadores_duales > 1e-4
)[0]

vectores_soporte = X_train_scaled[indices_soporte]
etiquetas_soporte = y_train[indices_soporte]

print("\nCantidad de vectores de soporte:", len(indices_soporte))
print("Índices de los vectores de soporte:", indices_soporte)


# ============================================================
# 12. GRÁFICA DE LA FRONTERA DE DECISIÓN
# ============================================================

# Se crea una cuadrícula para evaluar la función:
#
#             f(x) = w^T x + b

x_min = X_scaled[:, 0].min() - 1
x_max = X_scaled[:, 0].max() + 1

y_min = X_scaled[:, 1].min() - 1
y_max = X_scaled[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 500),
    np.linspace(y_min, y_max, 500)
)

puntos_malla = np.c_[xx.ravel(), yy.ravel()]

valores_decision = (
    puntos_malla @ w.value + b.value
).reshape(xx.shape)


plt.figure(figsize=(10, 7))

# Regiones de clasificación
plt.contourf(
    xx,
    yy,
    valores_decision,
    levels=[-100, 0, 100],
    alpha=0.20,
    cmap="coolwarm"
)

# Hiperplano y márgenes
contornos = plt.contour(
    xx,
    yy,
    valores_decision,
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    linewidths=[1.5, 2.5, 1.5]
)

plt.clabel(
    contornos,
    fmt={
        -1: "Margen -1",
        0: "Hiperplano",
        1: "Margen +1"
    }
)

# Datos completos
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=y,
    cmap="coolwarm",
    edgecolors="black",
    s=55,
    alpha=0.85
)

# Vectores de soporte
plt.scatter(
    vectores_soporte[:, 0],
    vectores_soporte[:, 1],
    s=190,
    facecolors="none",
    edgecolors="black",
    linewidths=2,
    label="Vectores de soporte"
)

plt.title(
    "Support Vector Machine mediante optimización convexa\n"
    f"Exactitud de prueba: {accuracy_test:.2%}"
)

plt.xlabel("Característica 1 normalizada")
plt.ylabel("Característica 2 normalizada")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


# ============================================================
# 13. MATRIZ DE CONFUSIÓN
# ============================================================

matriz = confusion_matrix(
    y_test,
    y_pred_test,
    labels=[-1, 1]
)

visualizacion = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Clase -1", "Clase +1"]
)

visualizacion.plot(
    values_format="d"
)

plt.title("Matriz de confusión de la SVM")
plt.show()


# ============================================================
# 14. PREDICCIÓN DE UN NUEVO PUNTO
# ============================================================

# Este punto se encuentra en las coordenadas originales,
# antes de la normalización.

nuevo_punto = np.array([[1.5, 1.8]])

nuevo_punto_scaled = escalador.transform(nuevo_punto)

prediccion_nueva = predecir(nuevo_punto_scaled)[0]

valor_decision_nuevo = (
    nuevo_punto_scaled @ w.value + b.value
)[0]

print("\n" + "=" * 60)
print("PREDICCIÓN DE UN NUEVO PUNTO")
print("=" * 60)

print("Nuevo punto:", nuevo_punto[0])
print("Valor de decisión:", valor_decision_nuevo)
print("Clase predicha:", prediccion_nueva)


# ============================================================
# 15. VERIFICACIÓN DE LAS RESTRICCIONES
# ============================================================

valores_restricciones = y_train * (
    X_train_scaled @ w.value + b.value
)

violaciones = np.maximum(
    0,
    1 - valores_restricciones
)

print("\n" + "=" * 60)
print("VERIFICACIÓN DEL PROBLEMA CONVEXO")
print("=" * 60)

print("Mayor violación del margen:", violaciones.max())
print("Suma de variables de holgura:", np.sum(xi.value))

print("\nEl programa terminó correctamente.")